Identify the relevant columns
Create the executed F&O orders table
 retain only executed orders.

In [0]:
%sql

CREATE OR REPLACE TABLE fno_successful AS

SELECT
    third_party_id,
    pt,
    order_created_at,
    order_updated_at,
    groww_order_id,
    contract_id,
    order_status,
    order_type,
    product,
    buy_sell,
    qty,
    filled_qty,
    avg_fill_price,
    platform,
    contract_type,
    call_put,
    symbol_name,
    contracts_underlying,
    contracts_expiry_date,
    is_basket_order,
    is_fast_exit,
    is_exitall,
    api_flag,
    is_tradingday,
    is_entry,
    pnl,
    position_exit_ts,
    exit_count,
    brokerage_inr

FROM fno_orders

WHERE UPPER(TRIM(order_status)) = 'EXECUTED';


Build user-level F&O metrics
we aggregate all executed orders by third_party_id.

We want to understand how much revenue each user generates and how frequently they trade.

In [0]:
%sql

CREATE OR REPLACE TABLE fno_user_metrics AS

SELECT

    third_party_id,

    -- Revenue
    SUM(COALESCE(brokerage_inr, 0))
        AS total_brokerage,

    -- Trading activity
    COUNT(DISTINCT groww_order_id)
        AS total_executed_orders,

    COUNT(DISTINCT TO_DATE(order_created_at))
        AS active_trading_days,

    SUM(COALESCE(filled_qty, 0))
        AS total_filled_qty,

    -- Trading diversity
    COUNT(DISTINCT contract_id)
        AS distinct_contracts,

    COUNT(DISTINCT contracts_underlying)
        AS distinct_underlyings,

    COUNT(DISTINCT contract_type)
        AS distinct_contract_types,

    COUNT(DISTINCT product)
        AS distinct_products,

    -- Trading behaviour
    SUM(CASE WHEN product = 'MIS'
        THEN 1 ELSE 0 END) AS mis_orders,

    SUM(CASE WHEN product = 'NRML'
        THEN 1 ELSE 0 END) AS nrml_orders,

    SUM(CASE WHEN contract_type = 'FUT'
        THEN 1 ELSE 0 END) AS futures_orders,

    SUM(CASE WHEN contract_type = 'OPT'
        THEN 1 ELSE 0 END) AS options_orders,

    SUM(CASE WHEN api_flag = 1
        THEN 1 ELSE 0 END) AS api_orders,

   SUM(CASE WHEN is_basket_order = TRUE
    THEN 1 ELSE 0 END) AS basket_orders,

    SUM(CASE WHEN is_entry = TRUE
        THEN 1 ELSE 0 END) AS entry_orders,

    SUM(CASE WHEN is_entry = TRUE
        THEN 1 ELSE 0 END) AS exit_orders,

    -- Trading outcomes
    SUM(COALESCE(pnl, 0))
        AS total_realized_pnl,

    MAX(order_created_at)
        AS last_order_date

FROM fno_successful

GROUP BY third_party_id;

Identify high-value F&O users
We'll initially use brokerage contribution to classify users.

In [0]:
%sql

CREATE OR REPLACE TABLE fno_user_segments AS

SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment

FROM fno_user_metrics;

Measure revenue concentration
Now we need to establish how much F&O revenue comes from the top 20%.

In [0]:
%sql

SELECT

    value_segment,

    COUNT(*) AS total_users,

    ROUND(AVG(total_brokerage), 2)
        AS avg_brokerage,

    ROUND(SUM(total_brokerage), 2)
        AS total_brokerage,

    ROUND(
        100.0 * SUM(total_brokerage) /
        NULLIF(SUM(SUM(total_brokerage)) OVER (), 0),
        2
    ) AS revenue_contribution_pct

FROM fno_user_segments

GROUP BY value_segment

ORDER BY value_segment;

Compare high-value users against other users
This will help us determine:
- Whether high-value users execute more orders.
- Whether they trade on more days.
- Whether they trade across more contracts and underlying instruments.
- Whether their realised trading outcomes differ.
Remember, P&L is a behavioural descriptor, not a measure of Groww's revenue

In [0]:
%sql

SELECT

    CASE
        WHEN value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    COUNT(*) AS total_users,

    ROUND(AVG(total_brokerage), 2)
        AS avg_brokerage,

    ROUND(AVG(total_executed_orders), 2)
        AS avg_orders,

    ROUND(AVG(active_trading_days), 2)
        AS avg_active_days,

    ROUND(AVG(distinct_contracts), 2)
        AS avg_contracts,

    ROUND(AVG(distinct_underlyings), 2)
        AS avg_underlyings,

    ROUND(AVG(total_realized_pnl), 2)
        AS avg_realized_pnl

FROM fno_user_segments

GROUP BY 1;